# CIFAR-10 Kernel-Scale Benchmark: VGG16 (Deep 3×3 Stack) vs. VGG10 (Large Kernels)

[![PyTorch](https://img.shields.io/badge/PyTorch-2.x-EE4C2C?logo=pytorch&logoColor=white)](https://pytorch.org/)
[![Repository](https://img.shields.io/badge/GitHub-vision--architectures--pytorch-181717?logo=github)](https://github.com/mohammadrezakasra/vision-architectures-pytorch)
[![Dataset](https://img.shields.io/badge/Dataset-CIFAR--10-blue)](https://www.cs.toronto.edu/~kriz/cifar.html)
[![Experiment](https://img.shields.io/badge/Focus-Receptive%20Field%20%26%20Kernel%20Depth-success)]()

> **Language:** [English](#1-overview) | [فارسی](#بخش-فارسی-بنچمارک-و-تحلیل-معماریهای-vgg-تقابل-هستههای-کوچک-و-بزرگ)

---

## 1. Overview

This notebook provides an empirical validation of the foundational **VGG hypothesis** (*Simonyan & Zisserman*) on **CIFAR-10**: evaluating whether a deep stack of small $3\times3$ convolutional filters (**VGG16**) generalizes better than a shallower hierarchy relying on large spatial kernels ($11\times11$, $7\times7$, $5\times5$) inspired by early AlexNet-style designs (**VGG10**). Beyond overall accuracy and loss, this study conducts a fine-grained **per-class accuracy breakdown** across all 10 CIFAR-10 categories to expose how kernel scale affects rigid geometric objects versus deformable fine-textured animals.

---

## 2. Evaluated Architectures

| Architecture | Spatial Filter Design | Classifier Head | Core Design Philosophy |
| :--- | :--- | :--- | :--- |
| **`VGG16` (Classic)** | Exclusive $3\times3$ Convolutional Stacks | Lightweight Funnel Classifier | Deep hierarchical non-linearity & parameter efficiency |
| **`VGG10` (Custom)** | Early Large Kernels ($11\times11 \rightarrow 7\times7 \rightarrow 5\times5$) | Identical Lightweight Funnel Classifier | Wide immediate receptive fields in shallow layers |

---

## 3. Experimental Setup & Training Pipeline

* **Classifier Adaptation for $32\times32$ Inputs:** To prevent severe overfitting on CIFAR-10 and isolate the effect of the convolutional backbone, standard bulky `4096`-unit dense layers are replaced with an identical, lightweight funnel classifier head across both networks.
* **Optimization & Regularization:** Both models are trained under identical hyperparameters using `AdamW` (`lr=1e-4`), unified data augmentation pipelines, and a stratified validation split.
* **Training Duration:** `100` epochs managed via the custom `Trainer` class with automated best-checkpoint saving (`./checkpoints/`).

---

## 4. Final Benchmark Results & Per-Class Breakdown (10,000 Unseen Test Images)

| Metric / Class | **`VGG16` (`3×3` Stack)** | **`VGG10` (Large Kernels)** | Delta (`VGG16` vs. `VGG10`) |
| :--- | :---: | :---: | :---: |
| **Overall Test Accuracy** | **89.30%** | **84.79%** | **+4.51%** |
| **Overall Test Loss** | **0.4219** | **0.5750** | **-0.1531** |
| `plane` | 89.90% | **94.00%** | -4.10% |
| `car` | **94.70%** | 93.80% | +0.90% |
| `bird` | **83.90%** | 75.50% | **+8.40%** |
| `cat` | **85.30%** | 65.90% | **+19.40%** |
| `deer` | **90.80%** | 83.40% | **+7.40%** |
| `dog` | **79.00%** | 77.90% | +1.10% |
| `frog` | **92.20%** | 90.60% | +1.60% |
| `horse` | **88.00%** | 86.50% | +1.50% |
| `ship` | **94.70%** | 91.80% | +2.90% |
| `truck` | **94.50%** | 88.50% | **+6.00%** |

---

## 5. Key Engineering Insights & Analysis

1. **Decisive Generalization Advantage (`+4.51%` Overall Accuracy):**
   `VGG16` achieves **89.30%** test accuracy and **0.4219** test loss, outperforming `VGG10` (**84.79%**, loss **0.5750**) by **+4.51%**. Factorizing large spatial filters into deep $3\times3$ cascades significantly improves representation capacity.
2. **Catastrophic Failure of Large Kernels on Fine-Grained Textures (`cat`: `85.30%` vs. `65.90%`):**
   The per-class breakdown reveals a striking pattern: `VGG10` performs competitively on rigid, high-contrast shapes with uniform backgrounds (such as `plane` at **94.00%** and `car` at **93.80%**), where global contours dominate. However, on deformable classes relying on subtle local fur and feather textures, `VGG10` collapses—dropping by **-19.40%** on `cat` (`65.90%` vs. `85.30%`), **-8.40%** on `bird`, and **-7.40%** on `deer`. Applying an $11\times11$ filter directly to a $32\times32$ image acts as a destructive spatial low-pass filter that blurs out high-frequency local discriminators.
3. **Compounded Non-Linearity at Equal Receptive Fields:**
   Three stacked $3\times3$ convolutional layers yield the exact same effective receptive field ($7\times7$) as a single $7\times7$ layer, yet inject **three sequential ReLU non-linearities** instead of one. This higher density of non-linear transformations enables `VGG16` to construct far more discriminative decision boundaries.
4. **Implicit Parameter Regularization:**
   While three $3\times3$ layers require $3 \times (3^2) C^2 = 27C^2$ weights, a single $7\times7$ layer requires $49C^2$ weights (and an $11\times11$ layer requires $121C^2$). Decomposing large kernels into $3\times3$ stacks acts as a structural regularizer that curbs overfitting on small-scale images.

---

## 6. Notebook Execution Flow

1. **Environment & Repository Setup:** Configuring PyTorch modules and custom utilities.
2. **CIFAR-10 Augmentation & Stratified Splitting:** Preparing training, validation, and test `DataLoader` pipelines.
3. **Architecture Instantiation (`VGG16` vs. `VGG10`):** Building both networks with matched lightweight classifier heads.
4. **100-Epoch Training Loop:** Training both models under identical `AdamW` (`lr=1e-4`) settings via `Trainer`.
5. **Test Set & Per-Class Evaluation:** Loading best checkpoints and computing overall and class-wise accuracy metrics.

---
---

<div dir="rtl" align="right" style="line-height: 1.8;">

# بخش فارسی: بنچمارک و تحلیل معماری‌های VGG: تقابل هسته‌های کوچک و بزرگ

## ۱. معرفی پروژه

در این نوت‌بوک، یک آزمایش کنترل‌شده برای ارزیابی تجربی فرضیه کلیدی معماری **VGG** (*Simonyan & Zisserman*) روی دیتاست **CIFAR-10** انجام شده است. هدف اصلی، مقایسه کارایی پشته‌ای عمیق از کانولوشن‌های کوچک $3\times3$ (**VGG16**) در برابر شبکه‌های کم‌عمق‌تر با هسته‌های بزرگ $11\times11$، $7\times7$ و $5\times5$ (**VGG10**، مشابه فلسفه AlexNet) است. علاوه بر دقت و خطای کلی، عملکرد هر دو معماری به تفکیک هر ۱۰ کلاس دیتاست تحلیل شده تا رفتار فیلترهای بزرگ و کوچک در مواجهه با اجسام صلب در برابر بافت‌های ظریف بررسی شود.

---

## ۲. معماری‌های مورد بررسی

| نام معماری | طراحی فیلترهای مکانی | بخش طبقه‌بند (Classifier) | فلسفه طراحی |
| :--- | :--- | :--- | :--- |
| **`VGG16` (کلاسیک)** | استفاده انحصاری از پشته‌های عمیق $3\times3$ | طبقه‌بند قیفی و سبک (Light) | افزایش چگالی توابع غیرخطی و بهینه‌سازی پارامترها |
| **`VGG10` (سفارشی)** | هسته‌های بزرگ ابتدایی ($11\times11 \rightarrow 7\times7 \rightarrow 5\times5$) | طبقه‌بند قیفی و سبک یکسان | میدان دید مکانی وسیع در لایه‌های کم‌عمق |

---

## ۳. طراحی آزمایش و خط لوله آموزش

* **طراحی طبقه‌بند متناسب با ابعاد $32\times32$:** برای ایجاد شرایط کاملاً عادلانه و جلوگیری از بیش‌برازش (Overfitting) روی تصاویر کوچک CIFAR-10، لایه‌های تمام‌متصل سنگین `4096` تایی حذف شده و هر دو مدل از یک Classifier قیفی و سبک (Light) با ابعاد کاملاً برابر استفاده می‌کنند تا صرفاً تفاوت استخراج ویژگی در لایه‌های کانولوشنی سنجیده شود.
* **پیکربندی یکپارچه آموزش:** هر دو مدل با تنظیمات کاملاً یکسان شامل بهینه‌ساز `AdamW`، نرخ یادگیری `0.0001` و تکنیک‌های افزونه‌سازی داده (Data Augmentation) یکپارچه آموزش دیده‌اند.
* **فرآیند اجرا:** آموزش با استفاده از کلاس کاستوم `Trainer` به همراه جداسازی اصولی `Validation Loader` و ذخیره‌سازی خودکار بهترین وزن‌ها (`Checkpoints`) به مدت **۱۰۰ ایپاک** انجام شده است.

---

## ۴. نتایج نهایی ارزیابی و تفکیک کلاس‌ها (۱۰,۰۰۰ تصویر تست دیده‌نشده)

| معیار / کلاس | **`VGG16` (پشته `3×3`)** | **`VGG10` (هسته‌های بزرگ)** | اختلاف عملکرد (`VGG16` نسبت به `VGG10`) |
| :--- | :---: | :---: | :---: |
| **دقت کل تست (Overall Accuracy)** | **89.30%** | **84.79%** | **+4.51%** |
| **خطای کل تست (Test Loss)** | **0.4219** | **0.5750** | **-0.1531** |
| `plane` (هواپیما) | 89.90% | **94.00%** | -4.10% |
| `car` (خودرو) | **94.70%** | 93.80% | +0.90% |
| `bird` (پرنده) | **83.90%** | 75.50% | **+8.40%** |
| `cat` (گربه) | **85.30%** | 65.90% | **+19.40%** |
| `deer` (گوزن) | **90.80%** | 83.40% | **+7.40%** |
| `dog` (سگ) | **79.00%** | 77.90% | +1.10% |
| `frog` (قورباغه) | **92.20%** | 90.60% | +1.60% |
| `horse` (اسب) | **88.00%** | 86.50% | +1.50% |
| `ship` (کشتی) | **94.70%** | 91.80% | +2.90% |
| `truck` (کامیون) | **94.50%** | 88.50% | **+6.00%** |

---

## ۵. تحلیل مهندسی و یافته‌های کلیدی

1. **برتری قاطع در تعمیم‌پذیری (`Generalization`):**
   مدل `VGG16` با ثبت دقت کل **۸۹.۳۰٪** و خطای تست **۰.۴۲۱۹** برتری معناداری نسبت به `VGG10` (دقت **۸۴.۷۹٪** و خطای **۰.۵۷۵۰**) نشان داد. این اختلاف **۴.۵۱ درصدی** ثابت می‌کند شکستن فیلترهای بزرگ به پشته‌های عمیق‌تر $3\times3$، ظرفیت شبکه را در استخراج سلسله‌مراتبی مفاهیم به شدت ارتقا می‌دهد.

2. **سقوط آزاد هسته‌های بزرگ در مواجهه با بافت‌های ظریف و پیچیده:**
   جدول تفکیک کلاس‌ها یک الگو بسیار مهم را آشکار می‌کند: فیلترهای بزرگ `VGG10` در کلاس‌های دارای هندسه صلب و پس‌زمینه یکنواخت (مانند `plane` با **۹۴.۰۰٪** و `car` با **۹۳.۸۰٪**) عملکرد خوبی دارند؛ اما در درک جزئیات ظریف حیوانات به شدت شکست می‌خورند. دقت کلاس «گربه» (`cat`) از **۸۵.۳۰٪** در `VGG16` به **۶۵.۹۰٪** در `VGG10` (**۱۹.۴۰٪ سقوط!**) و دقت کلاس «پرنده» (`bird`) به اندازه **۸.۴۰٪** کاهش یافته است؛ زیرا اعمال مستقیم فیلتر $11\times11$ روی تصویر کوچک $32\times32$ مانند یک فیلتر پایین‌گذر (Low-Pass) مخرب عمل کرده و بافت‌های فرکانس‌بالای مو و پر را در همان لایه اول محو می‌کند.

3. **مزیت چگالی توابع غیرخطی (`Non-Linearity Advantage`):**
   سه لایه متوالی $3\times3$ میدان دید مکانی مؤثر (Effective Receptive Field) کاملاً یکسانی با یک لایه $7\times7$ دارند؛ اما در `VGG16`، اطلاعات در این مسیر از **سه تابع فعال‌ساز غیرخطی `ReLU`** عبور می‌کنند. این چگالی بالاتر توابع غیرخطی، قدرت مدل را در تفکیک مرزهای تصمیم‌گیری پیچیده به شکل چشمگیری افزایش می‌دهد.

4. **صرفه‌جویی هوشمندانه در پارامترها و تنظیم‌کنندگی ذاتی:**
   معماری `VGG16` با وجود عمق بیشتر، به دلیل جایگزینی فیلترهای غول‌پیکر ($11\times11$ نیازمند $121C^2$ وزن و $7\times7$ نیازمند $49C^2$ وزن) با سه لایه $3\times3$ ($27C^2$ وزن)، پارامترهای کمتری در بلوک‌های معادل مصرف می‌کند. این ساختار فشرده‌تر خود به عنوان یک تنظیم‌کننده (Regularizer) طبیعی عمل کرده و مانع از بیش‌برازش می‌شود.

---

## ۶. ساختار اجرایی نوت‌بوک

1. **تنظیم محیط و فراخوانی ماژول‌ها:** آماده‌سازی ابزارهای پایتورچ و کلاس‌های مخزن.
2. **پیش‌پردازش و افزونه‌سازی CIFAR-10:** پیاده‌سازی Data Augmentation و جداسازی اصولی Validation Loader.
3. **ساخت معماری‌های `VGG16` و `VGG10`:** پیاده‌سازی دو شبکه با طبقه‌بندهای سبک هم‌اندازه.
4. **آموزش ۱۰۰ ایپاکی مدل‌ها:** اجرای حلقه آموزش یکپارچه با `AdamW` و ذخیره بهترین چک‌پوینت‌ها.
5. **ارزیابی کلان و تفکیک‌شده کلاس‌ها:** بارگذاری بهترین وزن‌ها و محاسبه دقت کلی و دقت تک‌تک ۱۰ کلاس روی داده‌های تست.

</div>

In [1]:
!git clone 'https://github.com/mohammadrezakasra/vision-architectures-pytorch.git'

Cloning into 'vision-architectures-pytorch'...
remote: Enumerating objects: 68, done.
remote: Counting objects: 100% (68/68), done.
remote: Compressing objects: 100% (52/52), done.
remote: Total 68 (delta 28), reused 47 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (68/68), 23.67 KiB | 2.37 MiB/s, done.
Resolving deltas: 100% (28/28), done.


In [2]:
import sys
sys.path.append('/content/vision-architectures-pytorch')

## 1. Imports & Setup


In [3]:
# !cd "/content/vision-architectures-pytorch"

In [24]:
import torch
import os
import torch.nn as nn
import torchvision
from torchvision import transforms
from sklearn.model_selection import train_test_split
import numpy as np
from torch.utils.data import TensorDataset , DataLoader ,Subset
from models.cnn.costom_vgg_10 import VGG10
from utils.trainer import Trainer
from models.cnn.vgg_16_classic import VGG16


In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [6]:
!sudo apt-get install -y aria2

!aria2c -x 16 -s 16 https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz -d ./data/

!tar -xvf ./data/cifar-10-python.tar.gz -C ./data/

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libaria2-0 libcares2
The following NEW packages will be installed:
  aria2 libaria2-0 libcares2
0 upgraded, 3 newly installed, 0 to remove and 33 not upgraded.
Need to get 1,566 kB of archives.
After this operation, 5,692 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble/main amd64 libcares2 amd64 1.27.0-1.0ubuntu1 [73.7 kB]
Get:2 http://archive.ubuntu.com/ubuntu noble/universe amd64 libaria2-0 amd64 1.37.0+debian-1build3 [1,105 kB]
Get:3 http://archive.ubuntu.com/ubuntu noble/universe amd64 aria2 amd64 1.37.0+debian-1build3 [387 kB]
Fetched 1,566 kB in 2s (892 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 79, <STDIN> line 3.)
debconf: falling b

In [7]:
## 2. Load Data & Exploratory Data Analysis (EDA)

train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(0.5),
    transforms.RandomCrop(32,padding = 4),
    transforms.ColorJitter(brightness= 0.2,contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))

])

eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
    ])


trainset = torchvision.datasets.CIFAR10(
    root="./data",
    train=True,
    download=False,
    transform=train_transform
)

valset  = torchvision.datasets.CIFAR10(
    root="./data",
    train=True,
    download=False,
    transform=eval_transform
)

testset = torchvision.datasets.CIFAR10(
    root="./data",
    train=False,
    download=False,
    transform=eval_transform
)

In [8]:
indices = list(range(len(trainset)))

In [9]:
train_ind , val_ind = train_test_split(indices,test_size=0.1,random_state=42,stratify=trainset.targets)

In [10]:
train_subset = Subset(trainset,train_ind)
val_subset = Subset(valset,val_ind)

In [11]:
train_loader = DataLoader(
    train_subset,
    batch_size=128,
    shuffle=True,
    num_workers=2
)

val_loader = DataLoader(
    val_subset,
    batch_size=128,
    shuffle=False,
    num_workers=2
)

test_loader = DataLoader(
    testset,
    batch_size=128,
    shuffle=False,
    num_workers=2
)

In [12]:
Vgg16 = VGG16(32,3,10).to(device)
vgg10 = VGG10(32,3,10).to(device)

In [16]:
optimizer_Vgg16 = torch.optim.AdamW(Vgg16.parameters(),lr=0.0001 ,weight_decay= 5e-4)
optimizer_Vgg10 = torch.optim.AdamW(vgg10.parameters(),lr=0.0001 ,weight_decay= 5e-4)

criterion = nn.CrossEntropyLoss()

In [17]:
train_vgg16 = Trainer(Vgg16,optimizer_Vgg16,criterion,device)
train_vgg10 = Trainer(vgg10,optimizer_Vgg10,criterion,device)

In [18]:
train_vgg10.fit(train_loader ,epochs=100 ,val_loader=val_loader ,model_name="vgg10")

Starting training on device: cuda for 100 epochs...

Epoch [01/100] | Train Loss: 1.7325 - Train Acc: 35.86% | Val Loss: 1.3324 - Val Acc: 52.06% --> (New Best! Saved to ../checkpoints/vgg10.pth)
Epoch [02/100] | Train Loss: 1.3174 - Train Acc: 52.91% | Val Loss: 1.2536 - Val Acc: 55.86% --> (New Best! Saved to ../checkpoints/vgg10.pth)
Epoch [03/100] | Train Loss: 1.1409 - Train Acc: 60.58% | Val Loss: 0.9909 - Val Acc: 64.86% --> (New Best! Saved to ../checkpoints/vgg10.pth)
Epoch [04/100] | Train Loss: 1.0203 - Train Acc: 64.95% | Val Loss: 0.9478 - Val Acc: 66.58% --> (New Best! Saved to ../checkpoints/vgg10.pth)
Epoch [05/100] | Train Loss: 0.9278 - Train Acc: 68.43% | Val Loss: 0.9328 - Val Acc: 67.20% --> (New Best! Saved to ../checkpoints/vgg10.pth)
Epoch [06/100] | Train Loss: 0.8575 - Train Acc: 70.94% | Val Loss: 0.7770 - Val Acc: 73.04% --> (New Best! Saved to ../checkpoints/vgg10.pth)
Epoch [07/100] | Train Loss: 0.7997 - Train Acc: 72.98% | Val Loss: 0.9165 - Val Acc: 69.

KeyboardInterrupt: 

In [20]:
train_vgg16.fit(train_loader ,epochs=70 ,val_loader=val_loader ,model_name="vgg16")

Starting training on device: cuda for 70 epochs...

Epoch [01/70] | Train Loss: 1.6726 - Train Acc: 37.67% | Val Loss: 1.1669 - Val Acc: 56.52% --> (New Best! Saved to ../checkpoints/vgg16.pth)
Epoch [02/70] | Train Loss: 1.1754 - Train Acc: 59.20% | Val Loss: 1.0496 - Val Acc: 64.44% --> (New Best! Saved to ../checkpoints/vgg16.pth)
Epoch [03/70] | Train Loss: 0.9591 - Train Acc: 67.26% | Val Loss: 0.8366 - Val Acc: 70.68% --> (New Best! Saved to ../checkpoints/vgg16.pth)
Epoch [04/70] | Train Loss: 0.8253 - Train Acc: 72.45% | Val Loss: 0.7128 - Val Acc: 75.96% --> (New Best! Saved to ../checkpoints/vgg16.pth)
Epoch [05/70] | Train Loss: 0.7282 - Train Acc: 75.85% | Val Loss: 0.7105 - Val Acc: 76.00% --> (New Best! Saved to ../checkpoints/vgg16.pth)
Epoch [06/70] | Train Loss: 0.6668 - Train Acc: 77.95% | Val Loss: 0.6523 - Val Acc: 78.18% --> (New Best! Saved to ../checkpoints/vgg16.pth)
Epoch [07/70] | Train Loss: 0.6033 - Train Acc: 80.05% | Val Loss: 0.6356 - Val Acc: 78.82% --> 

{'train_loss': [1.6726332456588746,
  1.1753766992357042,
  0.9590511402977837,
  0.8252917619917128,
  0.7282363958252801,
  0.6668230094485813,
  0.6033182687653436,
  0.5623987934536404,
  0.5181791104793548,
  0.48778629538748,
  0.45903780155181884,
  0.43333803831736245,
  0.4047335257424249,
  0.3960779441462623,
  0.3652639459821913,
  0.35146285184224446,
  0.3315511165883806,
  0.3175196709738837,
  0.3022204374631246,
  0.2915292432838016,
  0.2788815492524041,
  0.2638722473568387,
  0.25625818345281814,
  0.25200879322952696,
  0.23447442959149678,
  0.2257689510981242,
  0.21537822009192573,
  0.2117677117480172,
  0.1989008619599872,
  0.1905562920941247,
  0.18203860084745618,
  0.17610819126500024,
  0.17011327969233195,
  0.1658516105810801,
  0.16105040052202013,
  0.15239691269000372,
  0.14838769667413498,
  0.13975515166653527,
  0.1402255900700887,
  0.13170359728601244,
  0.12681381104257372,
  0.12698459068404302,
  0.11814889041052924,
  0.11985327739715576,
 

In [21]:
def evaluate_model(model, model_name, test_loader, criterion, device, checkpoint_dir="../checkpoints"):
    # ۱. جستجو و بارگذاری وزن‌های بهترین چک‌پوینت
    possible_paths = [
        os.path.join(checkpoint_dir, f"{model_name}.pth"),
        os.path.join("checkpoints", f"{model_name}.pth")
    ]

    loaded = False
    for path in possible_paths:
        if os.path.exists(path):
            model.load_state_dict(torch.load(path, map_location=device))
            print(f" Loaded best weights for {model_name} from: {path}")
            loaded = True
            break

    if not loaded:
        print(f" Checkpoint for {model_name} not found! Evaluating current in-memory weights...")

    model.eval()
    test_loss = 0.0
    correct = 0
    total = 0

    classes = ('plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')
    class_correct = [0] * 10
    class_total = [0] * 10

    # ۲. محاسبه شاخص‌ها روی تست
    with torch.no_grad():
        for inputs, targets in test_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, targets)

            test_loss += loss.item() * inputs.size(0)
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()

            c = (predicted == targets).squeeze()
            for i in range(targets.size(0)):
                label = targets[i].item()
                class_correct[label] += c[i].item()
                class_total[label] += 1

    avg_loss = test_loss / total
    overall_acc = 100.0 * correct / total

    per_class_acc = {
        classes[i]: (100.0 * class_correct[i] / class_total[i]) for i in range(10)
    }

    # ۳. نمایش گزارش اختصاصی مدل
    print("=" * 48)
    print(f"📊 Results for Model: {model_name.upper()}")
    print(f"📉 Test Loss:     {avg_loss:.4f}")
    print(f"🎯 Test Accuracy: {overall_acc:.2f}% ({correct}/{total})")
    print("=" * 48)

    return {
        "model_name": model_name,
        "loss": avg_loss,
        "acc": overall_acc,
        "per_class": per_class_acc
    }

In [25]:
# ارزیابی هر دو مدل روی داده‌های تست
res_vgg16 = evaluate_model(Vgg16, "vgg16", test_loader, criterion, device)
res_vgg10 = evaluate_model(vgg10, "vgg10", test_loader, criterion, device)

# چاپ جدول مقایسه رو‌در‌رو
print("\n" + "=" * 54)
print(f"{'Class / Metric':<15} | {'VGG16 (3x3 Stack)':<16} | {'VGG10 (Large Kernels)':<16}")
print("-" * 54)
print(f"{'Overall Acc':<15} | {res_vgg16['acc']:>14.2f}% | {res_vgg10['acc']:>19.2f}%")
print(f"{'Test Loss':<15} | {res_vgg16['loss']:>15.4f} | {res_vgg10['loss']:>20.4f}")
print("-" * 54)

classes = ('plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')
for cls in classes:
    acc16 = res_vgg16['per_class'][cls]
    acc10 = res_vgg10['per_class'][cls]
    print(f"{cls:<15} | {acc16:>14.2f}% | {acc10:>19.2f}%")
print("=" * 54)

 Loaded best weights for vgg16 from: ../checkpoints/vgg16.pth
📊 Results for Model: VGG16
📉 Test Loss:     0.4219
🎯 Test Accuracy: 89.30% (8930/10000)
 Loaded best weights for vgg10 from: ../checkpoints/vgg10.pth
📊 Results for Model: VGG10
📉 Test Loss:     0.5750
🎯 Test Accuracy: 84.79% (8479/10000)

Class / Metric  | VGG16 (3x3 Stack) | VGG10 (Large Kernels)
------------------------------------------------------
Overall Acc     |          89.30% |               84.79%
Test Loss       |          0.4219 |               0.5750
------------------------------------------------------
plane           |          89.90% |               94.00%
car             |          94.70% |               93.80%
bird            |          83.90% |               75.50%
cat             |          85.30% |               65.90%
deer            |          90.80% |               83.40%
dog             |          79.00% |               77.90%
frog            |          92.20% |               90.60%
horse           